<a href="https://colab.research.google.com/github/ekerzendorfer/MOL_LAB/blob/main/MOL_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title ▶ START MOL_LAB
# @markdown **1)** (Optional) SMILES unten anpassen  \n
# @markdown **2)** Dann links auf ▶️ **Run** klicken  \n
# @markdown Danach erscheint das Moleküllabor unten.

# @markdown ---
START_SMILES = "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"  # @param {type:"string"}
START_NAME = "Koffein"  # @param {type:"string"}
AUTO_LOAD_2D = True  # @param {type:"boolean"}
# @markdown ---

import sys, subprocess, os, stat, shutil, urllib.request, urllib.error

# ---- 1) Install (robust) ----
def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["py3Dmol", "ipywidgets"])

try:
    import rdkit
    print("✅ RDKit vorhanden:", rdkit.__version__)
except Exception:
    try:
        pip_install(["rdkit"])
        import rdkit
        print("✅ RDKit via pip:", rdkit.__version__)
    except Exception:
        print("⚠️ RDKit via pip nicht verfügbar → micromamba fallback ...")
        os.makedirs("/content/bin", exist_ok=True)
        subprocess.check_call("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content", shell=True)

        candidates = ["/content/bin/micromamba", "/content/bin/bin/micromamba", "/content/micromamba"]
        src = next((c for c in candidates if os.path.isfile(c)), None)
        if not src:
            raise RuntimeError("micromamba konnte nicht installiert werden (curl/tar).")

        dst = "/content/bin/micromamba"
        if src != dst:
            shutil.move(src, dst)
        os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

        prefix = "/content/rdkit_prefix"
        subprocess.check_call([dst, "create", "-y", "-p", prefix, "-c", "conda-forge", "python=3.10", "rdkit"],
                              stdout=subprocess.DEVNULL)

        site = f"{prefix}/lib/python3.10/site-packages"
        if site not in sys.path:
            sys.path.insert(0, site)

        import rdkit
        print("✅ RDKit via conda-forge:", rdkit.__version__)

# ---- 2) App-Code aus GitHub laden ----
USER = "ekerzendorfer"
REPO = "MOL_LAB"
PYFILE = "molekuellabor.py"

def try_download(branch):
    url = f"https://raw.githubusercontent.com/{USER}/{REPO}/{branch}/{PYFILE}"
    try:
        urllib.request.urlretrieve(url, PYFILE)
        print(f"✅ Geladen aus GitHub: {url}")
        return True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return False
        raise

ok = try_download("main") or try_download("master")
if not ok:
    raise FileNotFoundError(f"{PYFILE} nicht gefunden in {USER}/{REPO} (main/master).")

# ---- 3) Start-Parameter für die App verfügbar machen ----
# (Diese Variablen werden von molekuellabor.py gelesen, falls vorhanden.)
globals()["MOLLAB_START_SMILES"] = START_SMILES
globals()["MOLLAB_START_NAME"] = START_NAME
globals()["MOLLAB_AUTO_LOAD_2D"] = AUTO_LOAD_2D

# ---- 4) Start ----
exec(open(PYFILE, "r", encoding="utf-8").read())


✅ RDKit vorhanden: 2025.09.3
✅ Geladen aus GitHub: https://raw.githubusercontent.com/ekerzendorfer/MOL_LAB/main/molekuellabor.py
